# Communication Protocols Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Core Message Types

Every multi-agent system starts with a message format. We define types that map to what the real protocols use:

In [ ]:
```typescript

import crypto from "node:crypto";

type MessageRole = "user" | "agent";

type MessagePart =

  | { kind: "text"; text: string }

  | { kind: "data"; data: unknown; mediaType: string }

  | { kind: "file"; name: string; url: string; mediaType: string };

type TrajectoryEntry = {

  reasoning: string;

  toolName?: string;

  toolInput?: unknown;

  toolOutput?: unknown;

  timestamp: number;

};

type AgentMessage = {

  id: string;

  role: MessageRole;

  parts: MessagePart[];

  trajectory?: TrajectoryEntry[];

  replyTo?: string;

  timestamp: number;

};

function createMessage(

  role: MessageRole,

  parts: MessagePart[],

  replyTo?: string

): AgentMessage {

  return {

    id: crypto.randomUUID(),

    role,

    parts,

    replyTo,

    timestamp: Date.now(),

  };

}

function textMessage(role: MessageRole, text: string): AgentMessage {

  return createMessage(role, [{ kind: "text", text }]);

}

In [ ]:
```

Notice: `MessagePart` is multimodal (text, structured data, files) just like the real A2A and ACP specs. `TrajectoryEntry` captures the reasoning chain, matching ACP's TrajectoryMetadata.

### Step 2: A2A Agent Card and Registry

Build agent discovery that matches the real A2A spec:

In [ ]:
```typescript

type Skill = {

  id: string;

  name: string;

  description: string;

  tags: string[];

  inputModes: string[];

  outputModes: string[];

};

type AgentCard = {

  name: string;

  description: string;

  version: string;

  url: string;

  capabilities: {

    streaming: boolean;

    pushNotifications: boolean;

  };

  defaultInputModes: string[];

  defaultOutputModes: string[];

  skills: Skill[];

};

class AgentRegistry {

  private cards: Map<string, AgentCard> = new Map();

  register(card: AgentCard) {

    this.cards.set(card.name, card);

  }

  discoverBySkillTag(tag: string): AgentCard[] {

    return [...this.cards.values()].filter((card) =>

      card.skills.some((skill) => skill.tags.includes(tag))

    );

  }

  discoverByInputMode(mimeType: string): AgentCard[] {

    return [...this.cards.values()].filter(

      (card) =>

        card.defaultInputModes.includes(mimeType) ||

        card.skills.some((skill) => skill.inputModes.includes(mimeType))

    );

  }

  resolve(name: string): AgentCard | undefined {

    return this.cards.get(name);

  }

  listAll(): AgentCard[] {

    return [...this.cards.values()];

  }

}

In [ ]:
```

This is substantially richer than a simple name-to-capability map. You can discover agents by skill tags, by input MIME types, or by name, just like the real A2A spec supports.

### Step 3: A2A Task Lifecycle

Build the full task state machine:

In [ ]:
```typescript

type TaskState =

  | "submitted"

  | "working"

  | "input-required"

  | "auth-required"

  | "completed"

  | "failed"

  | "canceled"

  | "rejected";

const TERMINAL_STATES: TaskState[] = [

  "completed",

  "failed",

  "canceled",

  "rejected",

];

type TaskStatus = {

  state: TaskState;

  message?: AgentMessage;

  timestamp: number;

};

type Artifact = {

  id: string;

  name: string;

  parts: MessagePart[];

};

type Task = {

  id: string;

  contextId: string;

  status: TaskStatus;

  artifacts: Artifact[];

  history: AgentMessage[];

};

type TaskEvent =

  | { kind: "statusUpdate"; taskId: string; status: TaskStatus }

  | {

      kind: "artifactUpdate";

      taskId: string;

      artifact: Artifact;

      append: boolean;

      lastChunk: boolean;

    };

type TaskHandler = (

  task: Task,

  message: AgentMessage

) => AsyncGenerator<TaskEvent>;

class TaskManager {

  private tasks: Map<string, Task> = new Map();

  private handlers: Map<string, TaskHandler> = new Map();

  private listeners: Map<string, ((event: TaskEvent) => void)[]> = new Map();

  registerHandler(agentName: string, handler: TaskHandler) {

    this.handlers.set(agentName, handler);

  }

  subscribe(taskId: string, listener: (event: TaskEvent) => void) {

    const existing = this.listeners.get(taskId) ?? [];

    existing.push(listener);

    this.listeners.set(taskId, existing);

  }

  async sendMessage(

    agentName: string,

    message: AgentMessage,

    contextId?: string

  ): Promise<Task> {

    const handler = this.handlers.get(agentName);

    if (!handler) {

      const task = this.createTask(contextId);

      task.status = {

        state: "rejected",

        timestamp: Date.now(),

        message: textMessage("agent", `No handler for ${agentName}`),

      };

      return task;

    }

    const task = this.createTask(contextId);

    task.history.push(message);

    task.status = { state: "submitted", timestamp: Date.now() };

    this.processTask(task, handler, message).catch((err) => {

      task.status = {

        state: "failed",

        timestamp: Date.now(),

        message: textMessage("agent", String(err)),

      };

    });

    return task;

  }

  getTask(taskId: string): Task | undefined {

    return this.tasks.get(taskId);

  }

  cancelTask(taskId: string): boolean {

    const task = this.tasks.get(taskId);

    if (!task || TERMINAL_STATES.includes(task.status.state)) return false;

    task.status = { state: "canceled", timestamp: Date.now() };

    this.emit(taskId, {

      kind: "statusUpdate",

      taskId,

      status: task.status,

    });

    return true;

  }

  private createTask(contextId?: string): Task {

    const task: Task = {

      id: crypto.randomUUID(),

      contextId: contextId ?? crypto.randomUUID(),

      status: { state: "submitted", timestamp: Date.now() },

      artifacts: [],

      history: [],

    };

    this.tasks.set(task.id, task);

    return task;

  }

  private async processTask(

    task: Task,

    handler: TaskHandler,

    message: AgentMessage

  ) {

    task.status = { state: "working", timestamp: Date.now() };

    this.emit(task.id, {

      kind: "statusUpdate",

      taskId: task.id,

      status: task.status,

    });

    try {

      for await (const event of handler(task, message)) {

        if (TERMINAL_STATES.includes(task.status.state)) break;

        if (event.kind === "statusUpdate") {

          task.status = event.status;

        }

        if (event.kind === "artifactUpdate") {

          const existing = task.artifacts.find(

            (a) => a.id === event.artifact.id

          );

          if (existing && event.append) {

            existing.parts.push(...event.artifact.parts);

          } else {

            task.artifacts.push(event.artifact);

          }

        }

        this.emit(task.id, event);

      }

    } catch (err) {

      task.status = {

        state: "failed",

        timestamp: Date.now(),

        message: textMessage("agent", String(err)),

      };

      this.emit(task.id, {

        kind: "statusUpdate",

        taskId: task.id,

        status: task.status,

      });

    }

  }

  private emit(taskId: string, event: TaskEvent) {

    for (const listener of this.listeners.get(taskId) ?? []) {

      listener(event);

    }

  }

}

In [ ]:
```

This implements the real A2A task lifecycle: submitted, working, input-required, terminal states. Handlers are async generators that yield events (status updates and artifact chunks) matching the SSE streaming model.

### Step 4: ACP-Style Audit Trail

Wrap communication with trajectory tracking:

In [ ]:
```typescript

type AuditEntry = {

  runId: string;

  agentName: string;

  input: AgentMessage[];

  output: AgentMessage[];

  trajectory: TrajectoryEntry[];

  status: "created" | "in-progress" | "completed" | "failed" | "awaiting";

  startedAt: number;

  completedAt?: number;

  sessionId?: string;

};

class AuditableRunner {

  private log: AuditEntry[] = [];

  private handlers: Map<

    string,

    (input: AgentMessage[]) => Promise<{

      output: AgentMessage[];

      trajectory: TrajectoryEntry[];

    }>

  > = new Map();

  registerAgent(

    name: string,

    handler: (input: AgentMessage[]) => Promise<{

      output: AgentMessage[];

      trajectory: TrajectoryEntry[];

    }>

  ) {

    this.handlers.set(name, handler);

  }

  async run(

    agentName: string,

    input: AgentMessage[],

    sessionId?: string

  ): Promise<AuditEntry> {

    const entry: AuditEntry = {

      runId: crypto.randomUUID(),

      agentName,

      input: structuredClone(input),

      output: [],

      trajectory: [],

      status: "created",

      startedAt: Date.now(),

      sessionId,

    };

    this.log.push(entry);

    const handler = this.handlers.get(agentName);

    if (!handler) {

      entry.status = "failed";

      return entry;

    }

    entry.status = "in-progress";

    try {

      const result = await handler(input);

      entry.output = structuredClone(result.output);

      entry.trajectory = structuredClone(result.trajectory);

      entry.status = "completed";

      entry.completedAt = Date.now();

    } catch (err) {

      entry.status = "failed";

      entry.trajectory.push({

        reasoning: `Error: ${String(err)}`,

        timestamp: Date.now(),

      });

      entry.completedAt = Date.now();

    }

    return entry;

  }

  getFullAuditLog(): AuditEntry[] {

    return structuredClone(this.log);

  }

  getAuditLogForAgent(agentName: string): AuditEntry[] {

    return structuredClone(

      this.log.filter((e) => e.agentName === agentName)

    );

  }

  getAuditLogForSession(sessionId: string): AuditEntry[] {

    return structuredClone(

      this.log.filter((e) => e.sessionId === sessionId)

    );

  }

  getTrajectoryForRun(runId: string): TrajectoryEntry[] {

    const entry = this.log.find((e) => e.runId === runId);

    return entry ? structuredClone(entry.trajectory) : [];

  }

}

In [ ]:
```

Every agent execution produces a full audit entry: what went in, what came out, and the complete trajectory of tool calls and reasoning steps in between. You can query by agent, by session, or by individual run.

### Step 5: ANP-Style Identity Verification

Build DID-based identity and verification:

In [ ]:
```typescript

type VerificationMethod = {

  id: string;

  type: string;

  controller: string;

  publicKeyDer: string;

};

type DIDDocument = {

  id: string;

  verificationMethod: VerificationMethod[];

  authentication: string[];

  keyAgreement: string[];

  humanAuthorization: string[];

  service: { id: string; type: string; serviceEndpoint: string }[];

};

type AgentIdentity = {

  did: string;

  document: DIDDocument;

  privateKey: crypto.KeyObject;

  publicKey: crypto.KeyObject;

};

class IdentityRegistry {

  private documents: Map<string, DIDDocument> = new Map();

  publish(doc: DIDDocument) {

    this.documents.set(doc.id, doc);

  }

  resolve(did: string): DIDDocument | undefined {

    return this.documents.get(did);

  }

  verify(did: string, signature: string, payload: string): boolean {

    const doc = this.documents.get(did);

    if (!doc) return false;

    const authKeyIds = doc.authentication;

    const authKeys = doc.verificationMethod.filter((vm) =>

      authKeyIds.includes(vm.id)

    );

    for (const key of authKeys) {

      const publicKey = crypto.createPublicKey({

        key: Buffer.from(key.publicKeyDer, "base64"),

        format: "der",

        type: "spki",

      });

      const isValid = crypto.verify(

        null,

        Buffer.from(payload),

        publicKey,

        Buffer.from(signature, "hex")

      );

      if (isValid) return true;

    }

    return false;

  }

  requiresHumanAuth(did: string, operationKeyId: string): boolean {

    const doc = this.documents.get(did);

    if (!doc) return false;

    return doc.humanAuthorization.includes(operationKeyId);

  }

}

function createIdentity(domain: string, agentName: string): AgentIdentity {

  const did = `did:wba:${domain}:agent:${agentName}`;

  const { publicKey, privateKey } = crypto.generateKeyPairSync("ed25519");

  const publicKeyDer = publicKey

    .export({ format: "der", type: "spki" })

    .toString("base64");

  const keyId = `${did}#key-1`;

  const encKeyId = `${did}#key-x25519-1`;

  const document: DIDDocument = {

    id: did,

    verificationMethod: [

      {

        id: keyId,

        type: "Ed25519VerificationKey2020",

        controller: did,

        publicKeyDer,

      },

      {

        id: encKeyId,

        type: "X25519KeyAgreementKey2019",

        controller: did,

        publicKeyDer,

      },

    ],

    authentication: [keyId],

    keyAgreement: [encKeyId],

    humanAuthorization: [],

    service: [

      {

        id: `${did}#agent-description`,

        type: "AgentDescription",

        serviceEndpoint: `https://${domain}/agents/${agentName}/ad.json`,

      },

    ],

  };

  return { did, document, privateKey, publicKey };

}

function signPayload(identity: AgentIdentity, payload: string): string {

  return crypto

    .sign(null, Buffer.from(payload), identity.privateKey)

    .toString("hex");

}

In [ ]:
```

This mirrors the real ANP identity model: agents have DID documents with separate authentication, key agreement, and human authorization keys. The `IdentityRegistry` simulates DID resolution (in production this would be HTTP fetches to the agent's domain).

### Step 6: Protocol Gateway

Connect all four protocols into a unified system:

In [ ]:
```mermaid

graph LR

    REQ[Incoming Request] --> ANP_V{ANP: Verify DID}

    ANP_V -->|Valid| A2A_D{A2A: Discover Agent}

    ANP_V -->|Invalid| REJECT[Reject]

    A2A_D -->|Found| ACP_A[ACP: Audit Run]

    A2A_D -->|Not Found| REJECT

    ACP_A --> A2A_T[A2A: Create Task]

    A2A_T --> RESULT[Task + Audit Entry]

    style ANP_V fill:#d1fae5,stroke:#059669

    style A2A_D fill:#dbeafe,stroke:#2563eb

    style ACP_A fill:#fef3c7,stroke:#d97706

    style A2A_T fill:#dbeafe,stroke:#2563eb

In [ ]:
```

In [ ]:
```typescript

class ProtocolGateway {

  private registry: AgentRegistry;

  private taskManager: TaskManager;

  private auditRunner: AuditableRunner;

  private identityRegistry: IdentityRegistry;

  constructor(

    registry: AgentRegistry,

    taskManager: TaskManager,

    auditRunner: AuditableRunner,

    identityRegistry: IdentityRegistry

  ) {

    this.registry = registry;

    this.taskManager = taskManager;

    this.auditRunner = auditRunner;

    this.identityRegistry = identityRegistry;

  }

  async delegateTask(

    fromDid: string,

    signature: string,

    targetAgent: string,

    message: AgentMessage,

    sessionId?: string

  ): Promise<{ task: Task; audit: AuditEntry } | { error: string }> {

    if (!this.identityRegistry.verify(fromDid, signature, message.id)) {

      return { error: "Identity verification failed" };

    }

    const card = this.registry.resolve(targetAgent);

    if (!card) {

      return { error: `Agent ${targetAgent} not found in registry` };

    }

    const audit = await this.auditRunner.run(

      targetAgent,

      [message],

      sessionId

    );

    const task = await this.taskManager.sendMessage(targetAgent, message);

    return { task, audit };

  }

  discoverAndDelegate(

    fromDid: string,

    signature: string,

    skillTag: string,

    message: AgentMessage

  ): Promise<{ task: Task; audit: AuditEntry } | { error: string }> {

    const candidates = this.registry.discoverBySkillTag(skillTag);

    if (candidates.length === 0) {

      return Promise.resolve({

        error: `No agents found with skill tag: ${skillTag}`,

      });

    }

    return this.delegateTask(

      fromDid,

      signature,

      candidates[0].name,

      message

    );

  }

}

In [ ]:
```

The gateway does four things in one call:

1. **ANP**: Verifies the caller's identity via DID signature

2. **A2A**: Discovers the target agent and checks capabilities

3. **ACP**: Wraps the execution in an audit trail with trajectory

4. **A2A**: Creates a task with full lifecycle tracking

### Step 7: Wire It All Together

In [ ]:
```typescript

async function protocolDemo() {

  const registry = new AgentRegistry();

  registry.register({

    name: "researcher",

    description: "Searches and summarizes findings",

    version: "1.0.0",

    url: "https://researcher.local/a2a/v1",

    capabilities: { streaming: true, pushNotifications: false },

    defaultInputModes: ["text/plain"],

    defaultOutputModes: ["text/plain", "application/json"],

    skills: [

      {

        id: "web-research",

        name: "Web Research",

        description: "Searches the web",

        tags: ["research", "search", "summarization"],

        inputModes: ["text/plain"],

        outputModes: ["application/json"],

      },

    ],

  });

  registry.register({

    name: "coder",

    description: "Writes code from specs",

    version: "1.0.0",

    url: "https://coder.local/a2a/v1",

    capabilities: { streaming: false, pushNotifications: false },

    defaultInputModes: ["text/plain", "application/json"],

    defaultOutputModes: ["text/plain"],

    skills: [

      {

        id: "code-gen",

        name: "Code Generation",

        description: "Generates code",

        tags: ["coding", "generation"],

        inputModes: ["text/plain", "application/json"],

        outputModes: ["text/plain"],

      },

    ],

  });

  const taskManager = new TaskManager();

  const auditRunner = new AuditableRunner();

  const researchTrajectory: TrajectoryEntry[] = [];

  taskManager.registerHandler(

    "researcher",

    async function* (task, message) {

      yield {

        kind: "statusUpdate" as const,

        taskId: task.id,

        status: { state: "working" as const, timestamp: Date.now() },

      };

      researchTrajectory.push({

        reasoning: "Searching for React 19 documentation",

        toolName: "web_search",

        toolInput: { query: "React 19 compiler features" },

        toolOutput: {

          results: ["react.dev/blog/react-19", "github.com/react/react"],

        },

        timestamp: Date.now(),

      });

      researchTrajectory.push({

        reasoning: "Extracting key findings from search results",

        toolName: "doc_analysis",

        toolInput: { url: "react.dev/blog/react-19" },

        toolOutput: {

          summary:

            "React 19 compiler auto-memoizes, no manual useMemo needed",

        },

        timestamp: Date.now(),

      });

      yield {

        kind: "artifactUpdate" as const,

        taskId: task.id,

        artifact: {

          id: crypto.randomUUID(),

          name: "research-results",

          parts: [

            {

              kind: "data" as const,

              data: {

                findings: [

                  "React 19 compiler auto-memoizes components",

                  "No more manual useMemo/useCallback needed",

                  "Compiler runs at build time, not runtime",

                ],

                sources: ["react.dev/blog/react-19"],

              },

              mediaType: "application/json",

            },

          ],

        },

        append: false,

        lastChunk: true,

      };

      yield {

        kind: "statusUpdate" as const,

        taskId: task.id,

        status: { state: "completed" as const, timestamp: Date.now() },

      };

    }

  );

  auditRunner.registerAgent("researcher", async () => ({

    output: [

      textMessage("agent", "React 19 compiler auto-memoizes components"),

    ],

    trajectory: researchTrajectory,

  }));

  const identityRegistry = new IdentityRegistry();

  const coderIdentity = createIdentity("coder.local", "coder");

  const researcherIdentity = createIdentity("researcher.local", "researcher");

  identityRegistry.publish(coderIdentity.document);

  identityRegistry.publish(researcherIdentity.document);

  const gateway = new ProtocolGateway(

    registry,

    taskManager,

    auditRunner,

    identityRegistry

  );

  console.log("=== Protocol Demo ===\n");

  console.log("1. Agent Discovery (A2A)");

  const researchAgents = registry.discoverBySkillTag("research");

  console.log(

    `   Found ${researchAgents.length} agent(s):`,

    researchAgents.map((a) => a.name)

  );

  console.log("\n2. Identity Verification (ANP)");

  const message = textMessage("user", "Research React 19 compiler features");

  const signature = signPayload(coderIdentity, message.id);

  const verified = identityRegistry.verify(

    coderIdentity.did,

    signature,

    message.id

  );

  console.log(`   Coder DID: ${coderIdentity.did}`);

  console.log(`   Signature verified: ${verified}`);

  console.log("\n3. Task Delegation (A2A + ACP + ANP)");

  const result = await gateway.delegateTask(

    coderIdentity.did,

    signature,

    "researcher",

    message,

    "session-001"

  );

  if ("error" in result) {

    console.log(`   Error: ${result.error}`);

    return;

  }

  console.log(`   Task ID: ${result.task.id}`);

  console.log(`   Task state: ${result.task.status.state}`);

  console.log(`   Artifacts: ${result.task.artifacts.length}`);

  console.log("\n4. Audit Trail (ACP)");

  console.log(`   Run ID: ${result.audit.runId}`);

  console.log(`   Status: ${result.audit.status}`);

  console.log(`   Trajectory steps: ${result.audit.trajectory.length}`);

  for (const step of result.audit.trajectory) {

    console.log(`     - ${step.reasoning}`);

    if (step.toolName) {

      console.log(`       Tool: ${step.toolName}`);

    }

  }

  console.log("\n5. Full Audit Log");

  const fullLog = auditRunner.getFullAuditLog();

  console.log(`   Total runs: ${fullLog.length}`);

  for (const entry of fullLog) {

    const duration = entry.completedAt

      ? `${entry.completedAt - entry.startedAt}ms`

      : "in-progress";

    console.log(`   ${entry.agentName}: ${entry.status} (${duration})`);

  }

}

protocolDemo().catch((err) => {

  console.error("Protocol demo failed:", err);

  process.exitCode = 1;

});

In [ ]:
```

## Exercises

In [ ]:
1. **Multi-hop task delegation.** Extend the `TaskManager` so an agent handler can delegate subtasks to other agents. The researcher receives a task, delegates "search" and "summarize" subtasks to two specialist agents, waits for both to complete, then merges the results into its own artifacts.

2. **Streaming audit trail.** Modify the `AuditableRunner` to support streaming mode. Instead of waiting for the full result, yield `AuditEntry` updates in real-time as trajectory entries are added. Use an async generator that produces audit snapshots.

3. **DID rotation.** Add key rotation to the `IdentityRegistry`. An agent should be able to publish a new DID document with updated keys while maintaining a `previousDid` reference. Verifiers should accept signatures from both the current and previous key during a grace period.

4. **Protocol negotiation.** Implement ANP's meta-protocol concept. Two agents exchange `protocolNegotiation` messages with candidate formats (e.g., "I can speak JSON-RPC" vs "I prefer REST"). After max 3 rounds, they agree on a format or timeout. The agreed format determines which `TaskManager` or `AuditableRunner` they use.

5. **Rate-limited discovery.** Add a `RateLimitedRegistry` wrapper that caches Agent Card lookups with a configurable TTL and limits discovery queries per agent per second. Simulate a thundering herd of 100 agents discovering each other on startup and measure the difference.